# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, avg, median, col, coalesce, lit, when

# Read bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info_raw")
df.limit(50).display()

In [0]:
df.printSchema()

# Silver transformations

### Rename columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_number",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date",
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks

In [0]:
df.limit(5).display()

### Replace NULL in product_line with R.


In [0]:
df = df.fillna({"product_line": "R"})

In [0]:
df.limit(5).display()

### Replace NULL in product_cost to the median cost.

In [0]:
median_cost = df.select(median("product_cost")).first()[0]
df = df.fillna({"product_cost": median_cost})
df.limit(5).display()

### Replace NULL in product_end_date with infinity date.

In [0]:
df = df.withColumn("product_end_date",
    when
    (
        col("product_end_date").isNull(),
        lit("9999-12-31").cast("date")
    ).otherwise(col("product_end_date"))
)

df.limit(50).display()

# Write into Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_products
LIMIT 10